# QLoRA Fine-tuning TinyLlama on CNN/DailyMail
This notebook fine-tunes a generative LLM using PEFT (QLoRA) to generate news highlights.

In [1]:

!pip install -q transformers datasets peft bitsandbytes accelerate rouge-score trl


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 517.4/517.4 kB 19.1 MB/s eta 0:00:00


In [3]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.7 MB/s eta 0:00:00


In [4]:

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, Trainer, TrainingArguments
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
import evaluate


In [5]:

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [15]:
from datasets import load_dataset

raw_dataset = load_dataset("cnn_dailymail", "3.0.0", split="train[:1%]")


In [16]:
def preprocess(example):
    prompt = f"Summarize the following news article:\n{example['article']}\nSummary:\n"
    summary = example["highlights"]

    full_text = prompt + summary

    tokenized = tokenizer(
        full_text,
        truncation=True,
        padding="max_length",
        max_length=512
    )

    labels = tokenized["input_ids"].copy()

    prompt_len = len(
        tokenizer(prompt, truncation=True, max_length=512)["input_ids"]
    )

    labels[:prompt_len] = [-100] * prompt_len

    tokenized["labels"] = labels
    return tokenized


In [17]:
dataset = raw_dataset.map(
    preprocess,
    remove_columns=raw_dataset.column_names
)


Map:   0%|          | 0/2871 [00:00<?, ? examples/s]

In [18]:
len(dataset[0]["input_ids"]), len(dataset[0]["labels"])


(512, 512)

In [19]:

lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:282: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [20]:

training_args = TrainingArguments(
    output_dir="./qlora-cnn",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=1,
    fp16=True,
    logging_steps=10,
    save_steps=100,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
)

trainer.train()


Step,Training Loss
10,23.056800
20,63.884700
30,4.108900
40,5.025200
50,1.513500
60,3.655600
70,1.926500
80,2.330200
90,1.196600
100,0.853800


TrainOutput(global_step=718, training_loss=4.383311080235293, metrics={'train_runtime': 810.9504, 'train_samples_per_second': 3.54, 'train_steps_per_second': 0.885, 'total_flos': 9134035810910208.0, 'train_loss': 4.383311080235293, 'epoch': 1.0})

In [21]:

rouge = evaluate.load("rouge")

sample = load_dataset("cnn_dailymail", "3.0.0", split="test[:5]")

def generate_summary(article):
    inputs = tokenizer(article, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=80)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

preds, refs = [], []
for ex in sample:
    pred = generate_summary(ex["article"])
    preds.append(pred)
    refs.append(ex["highlights"])

rouge.compute(predictions=preds, references=refs)


{'rouge1': np.float64(0.16479688721176305),
 'rouge2': np.float64(0.08911118681828037),
 'rougeL': np.float64(0.12110839544857492),
 'rougeLsum': np.float64(0.1298685414782617)}


## Final Analysis

- The model learns to identify key news content.
- QLoRA significantly reduces trainable parameters.
- ROUGE scores show reasonable summarization quality given limited data.
